# Prepare Grouped Data for Task 5 - 3 SKU Groups (Fast/Medium/Slow 73 SKU)
Chia 220 SKU thành 3 nhóm theo MeanDemand từ `data/train.tfrecords` (220 SKU, 1000 periods).
- Fast: Top 73 SKU demand cao (SKU57 22.6...), turnover nhanh
- Medium: Middle 73 SKU (SKU100 6.25...)
- Slow: Bottom 74 SKU demand thấp + CV cao (SKU64 0.66 CV1.46)
Mỗi nhóm sẽ sinh `train.tfrecords` 1000 periods x 73, `capacity.tfrecords` 73 riêng trong `data_grouped/group_*/`


In [1]:
import tensorflow as tf, os, numpy as np, pathlib
BASE = r'C:\GitHub\Q-learning-for-Inventory-Management'
TRAIN_220 = os.path.join(BASE, 'data', 'train.tfrecords')
def read_sales_220(path):
    ds=tf.data.TFRecordDataset(path)
    sales=[]
    for rec in ds:
        ex=tf.train.Example(); ex.ParseFromString(rec.numpy())
        s=list(ex.features.feature['sales'].float_list.value)
        sales.append(s)
    return np.array(sales, dtype=np.float32)
train_sales_220 = read_sales_220(TRAIN_220)
mean_demand = train_sales_220.mean(axis=0)
sorted_idx = np.argsort(mean_demand)[::-1]
group_fast_idx = sorted_idx[:73]
group_medium_idx = sorted_idx[73:146]
group_slow_idx = sorted_idx[146:]
print(f'Fast Top 3 Mean: {[mean_demand[i] for i in group_fast_idx[:3]]}')
print(f'Medium Top 3 Mean: {[mean_demand[i] for i in group_medium_idx[:3]]}')
print(f'Slow Bottom 3 Mean: {[mean_demand[i] for i in group_slow_idx[-3:]]}')
print(f'Fast {len(group_fast_idx)}, Medium {len(group_medium_idx)}, Slow {len(group_slow_idx)}')


Fast Top 3 Mean: [22.625, 17.262, 15.567]
Medium Top 3 Mean: [1.714, 1.707, 1.677]
Slow Bottom 3 Mean: [0.323, 0.287, 0.28]
Fast 73, Medium 73, Slow 74


In [2]:
import sys
sys.path.append(r'C:\GitHub\Q-learning-for-Inventory-Management')
from prepare_data import sales_example, capacity_example, stock_example
import tensorflow as tf, numpy as np, os, pathlib, math
BASE = r'C:\GitHub\Q-learning-for-Inventory-Management'
def parse_cap(s):
    feat=tf.io.parse_single_example(s, {'capacity': tf.io.FixedLenFeature([220], tf.float32)})
    return feat['capacity']
def parse_stock(s):
    feat=tf.io.parse_single_example(s, {'stock': tf.io.FixedLenFeature([220], tf.float32)})
    return feat['stock']
def read_sales(path):
    ds=tf.data.TFRecordDataset(path)
    sales=[]
    for rec in ds:
        ex=tf.train.Example(); ex.ParseFromString(rec.numpy())
        s=list(ex.features.feature['sales'].float_list.value)
        sales.append(s)
    return np.array(sales, dtype=np.float32)
cap_220 = next(iter(tf.data.TFRecordDataset(os.path.join(BASE, 'data/capacity.tfrecords')).map(parse_cap))).numpy()
stock_220 = next(iter(tf.data.TFRecordDataset(os.path.join(BASE, 'data/stock.tfrecords')).map(parse_stock))).numpy()
train_220 = read_sales(os.path.join(BASE, 'data/train.tfrecords'))
test_220 = read_sales(os.path.join(BASE, 'data/test.tfrecords'))
def create_grouped(group_name, group_idx, n):
    cap_g = cap_220[group_idx]
    stock_g = stock_220[group_idx]
    train_g = train_220[:, group_idx]
    test_g = test_220[:, group_idx]
    out_dir = pathlib.Path(f'C:/GitHub/Q-learning-for-Inventory-Management/Feedback 7-9/task12-9/task 5/data_grouped/group_{group_name.lower()}')
    out_dir.mkdir(parents=True, exist_ok=True)
    with tf.io.TFRecordWriter(str(out_dir / 'capacity.tfrecords')) as w:
        w.write(capacity_example(cap_g).SerializeToString())
    with tf.io.TFRecordWriter(str(out_dir / 'stock.tfrecords')) as w:
        w.write(stock_example(stock_g).SerializeToString())
    with tf.io.TFRecordWriter(str(out_dir / 'train.tfrecords')) as w:
        for t in range(train_g.shape[0]):
            w.write(sales_example(train_g[t]).SerializeToString())
    with tf.io.TFRecordWriter(str(out_dir / 'test.tfrecords')) as w:
        for t in range(test_g.shape[0]):
            w.write(sales_example(test_g[t]).SerializeToString())
    print(f'Wrote {out_dir} {n} SKU')
for name, idx, n in [('Fast', group_fast_idx, 73), ('Medium', group_medium_idx, 73), ('Slow', group_slow_idx, 74)]:
    create_grouped(name, idx, n)


Wrote C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\data_grouped\group_fast 73 SKU
Wrote C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\data_grouped\group_medium 73 SKU
Wrote C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\data_grouped\group_slow 74 SKU


In [3]:
# Xuất thêm CSV cho từng nhóm để kiểm tra bằng Excel (như yêu cầu)
import pandas as pd
for group_name in ['Fast', 'Medium', 'Slow']:
    gdir = pathlib.Path(f'C:/GitHub/Q-learning-for-Inventory-Management/Feedback 7-9/task12-9/task 5/data_grouped/group_{group_name.lower()}')
    # Đọc lại TFRecords vừa tạo và xuất CSV
    import tensorflow as tf, numpy as np
    def read_sales_csv(path):
        ds=tf.data.TFRecordDataset(str(path))
        sales=[]
        for rec in ds:
            ex=tf.train.Example(); ex.ParseFromString(rec.numpy())
            s=list(ex.features.feature['sales'].float_list.value)
            sales.append(s)
        return np.array(sales)
    train = read_sales_csv(gdir / 'train.tfrecords')
    pd.DataFrame(train).to_csv(gdir / 'train.csv', index=False, header=False)
    print(f'Exported {gdir}/train.csv shape {train.shape}')


Exported C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\data_grouped\group_fast/train.csv shape (1000, 73)
Exported C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\data_grouped\group_medium/train.csv shape (1000, 73)
Exported C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\data_grouped\group_slow/train.csv shape (1000, 74)
